# 18. The regions that sold nothing, according to the join

**Track:** debug-the-ai | **Difficulty:** medium | **Tags:** joins, null keys, eqNullSafe, composite key

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("018-join-on-nullable-key")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "sales": """
UEFSMRUEFaAeFa4PTBXkAxUAEgAAkA8McREBAAEBAHINCABzDQgAdA0IAHUNCAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgA
fQ0IAH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACIDQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0I
AJANCACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgAmw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKIN
CACjDQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0IAK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1
DQgAtg0IALcNCAC4DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMANCADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgA
yA0IAMkNCADKDQgAyw0IAMwNCADNDQgAzg0IAM8NCADQDQgA0Q0IANINCADTDQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0I
ANsNCADcDQgA3Q0IAN4NCADfDQgA4A0IAOENCADiDQgA4w0IAOQNCADlDQgA5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IAO0N
CADuDQgA7w0IAPANCADxDQgA8g0IAPMNCAD0DQgA9Q0IAPYNCAD3DQgA+A0IAPkNCAD6DQgA+w0IAPwNCAD9DQgA/g0IAP8NCAQA
Eol4AAENCAACDQgAAw0IAAQNCAAFDQgABg0IAAcNCAAIDQgACQ0IAAoNCAALDQgADA0IAA0NCAAODQgADw0IABANCAARDQgAEg0I
ABMNCAAUDQgAFQ0IABYNCAAXDQgAGA0IABkNCAAaDQgAGw0IABwNCAAdDQgAHg0IAB8NCAAgDQgAIQ0IACINCAAjDQgAJA0IACUN
CAAmDQgAJw0IACgNCAApDQgAKg0IACsNCAAsDQgALQ0IAC4NCAAvDQgAMA0IADENCAAyDQgAMw0IADQNCAA1DQgANg0IADcNCAA4
DQgAOQ0IADoNCAA7DQgAPA0IAD0NCAA+DQgAPw0IAEANCABBDQgAQg0IAEMNCABEDQgARQ0IAEYNCABHDQgASA0IAEkNCABKDQgA
Sw0IAEwNCABNDQgATg0IAE8NCABQDQgAUQ0IAFINCABTDQgAVA0IAFUNCABWDQgAVw0IAFgNCABZDQgAWg0IAFsNCABcDQgAXQ0I
AF4NCABfDQgAYA0IPGESAQAAAAAAYhIBAAAAAAAVABWCBBWMBCwV5AMVEBUGFQYcGAhiEgEAAAAAABgIcREBAAAAAAAWACgIYhIB
AAAAAAAYCHERAQAAAAAAEREAAACBAvQAAQMAAADkAwEIPwABAgMEBQYHCAkKCwwNDg8QERITFBUWFxgZGhscHR4fICEiIyQlJico
KSorLC0uLzAxMjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldYWVpbXF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJz
dHV2d3h5ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp+goaKjpKWmp6ipqqusra6vsLGys7S1tre4ubq7vL2+
v8DBwsPExcbHyMnKy8zNzs/Q0dLT1NXW19jZ2tvc3d7f4OHi4+Tl5ufo6err7O3u7/DxAAAAAAAAFQQVVBVSTBUOFQASAAAqLAIA
AABJTgIAAABMSwEGBFVTAQYEQUUBBjRTRwIAAABCRAIAAABOUBUAFcwBFdIBLBXkAxUQFQYVBhw2ACgCVVMYAkFFEREAAABm8GUD
AAAA5AMBAz9AJI0QAKiQRABEIEAQBBCbrUEiClmwBGgEJhAdJGDgABggAALSoDgCFFOYBACSAEAQAEKCAGDMNGLAQAnSEEiEYQyS
IBUSARGSiFYBNACGCMCQAACQBAkgSKoQAAAVBBVgFVpMFRAVABIAADAUAgAAAFROAQYETUgBBgRUWAEGBFdBAQYETlkBBjRETAIA
AABDQQIAAABLQRUAFdYBFdwBLBXkAxUQFQYVBhw2igEoAldBGAJDQRERAAAAa/BqIwAAAB87f+/6v9C1fbZ8ud0bj/0aARf+Q1pf
V/3dvXS3+xgBBaIBAy2IxvoNFJlJWyO6IwQE/VOnfv1+YbU5+hY5zMuBBTFIaOMIwRznYDT1k9CZf/NptsWul/smY5xIouSn0vwx
3a04bwAVBBXqEhX0EkwV4gMVABIAALUJ9LQEAAAryvAAAEPUdAAADoycAAA6lOAAADyKcAAAFeqMAAAW6aAAAEuoXAAALByUAAAi
n6wAAFOChAAALKWwAABRshAAABG6rAAAEpfEAAA3KFAAAE3DCAAAVtYUAABPa6QAADJ4tAAAPVM4AAA+H4QAACMhwAAAGL38AAAZ
fpAAAD0K9AAAWTr4AABKpWAAAC++qAAAVO9cAAAKIVwAABC4FAAAI2poAABYIhwAADSMvAAAQ0UYAAAkkVQAADop2AAAIk+YAAAj
bSQAAC0FAAAAPZDwAABQ0dgAAB+3JAAAR+nEAAAHr5QAAEFdNAAAIedMAABAWKgAACH3tAAANZIQAABJmpQAACuG+AAAUlSQAABa
y8AAADHYKAAAMePgAAAQMbQAADWDnAAAF6BwAAAdKNgAACNynAAACZOQAABUfOgAAAhHJAAAKUnsAABDhfAAAB2XAAAAFy00AAAk
+mgAABfkzAAAIiJIAAA6ZTgAAEV7HAAAJxiYAAAMVvwAAEhulAAARIrgAAA4fVQAABHg9AAAEsiYAABVffAAAFg2CAAALaP8AAAc
lGgAAA1r8AAALZ7oAAAiU4AAAE+YkAAATc+IAABHMpAAAE87NAAAEkT0AABNcWQAAFqRjAAALZLMAAAdj5QAAFPhDAAAKgY0AABT
NMgAAFQ2NAAAJmFkAAA4OiQAABy56AAAHawYAABOFdgAABzkGAAAFHfYAAAPgSQAAFK3ZAAARl+gAAAeaxwAABfIrAAAOPscAAAi
HfwAAEtMkAAAJPI0AAAVlQAAAB7k/AAAUiCcAABMcewAAB6IaAAAPtUoAAAUHHAAAAw2LAAAHfMwAABJLAgAAB7ogAAAQhZcAABC
rYgAACIUnAAAScFAAAA6ZZwAABeYPAAAK9PsAAAN+0wAAA6VmAAADTjEAABGfhgAAEcgNAAAR5JEAAAOxNwAABLQzAAAT7OEAAA8
5jwAACV4+AAAImcIAABHhigAAE/WrAAAKadIAAAbhOwAAApIbAAAUci4AABGKIwAAD4qdAAAExWMAAA/w9QAACgOsAAAHX4AAAAg
xRAAAFnPBAAARIaUAABT+nAAABQ0RAAAWJI4AABUdeAAABWXWAAAB/PwAAAQ51gAAEBdWAAASJ3YAAAfktAAACAb7AAAPKoUAAAb
g1wAAC4bIAAAQzfQAAAIxuAAABBShAAACtacAAA12rgAAERJpAAAKygMAAA/UvAAADZvKAAANQBcAAA/wLQAABHZiAAAF/fwAAA5
r/gAAC17XAAAJyyEAAAvkbwAAEH7aAAAMxrQAABCjkgAABgkFAAAPGVUAAAOP0QAAFOwOAAAGttkAAA9QHgAADmNNAAAKAssAAA0
+CgAAFFGpAAACpugAAAr7VAAADHSTAAAS61wAABHubgAABD/kAAAEsMgAABYVzwAAFhOQAAAUJn8AAA9hsgAABV2JAAAIB5EAABb
CkAAAArOaAAAC/k8AAAPglAAADGUlAAAVTpcAAAvMUAAAEwW6AAAS/uQAAAtykQAAD7zoAAAUIrAAAAPm7QAACmlVAAAEUzoAABL
zLAAABdx9AAAU9L8AAAgSmgAAD6bvAAARAA0AABHoRwVABWCBBWMBCwV5AMVEBUGFQYcGAUAAFsKQBgFAAAHr5QWACgFAABbCkAY
BQAAB6+UEREAAACBAvQAAQMAAADkAwEIPwABAgMEBQYHCAkKCwwNDg8QERITFBUWFxgZGhscHR4fICEiIyQlJicoKSorLC0uLzAx
MjM0NTY3ODk6Ozw9Bj4/QEFCQ0RFRkdISUpLTE1OT1BRUlNUVVZXWFlaW1xdXl9gYWJjZGVmZ2hpamtsbW5vcHFyc3R1dnd4eXp7
fH1+f4CBgoOEhYaHiImKi4yNjo+QkZKTlJWWl5iZmpucnZ6foKGio6SlpqeoqaqrrK2ur7CxsrO0tba3uLm6u7y9vr/AwcLDxMXG
x8jJysvMzc7P0NHS09TV1tfY2drb3N3e3+Dh4uPk5ebn6Onq6+zt7u/wAAAAAAAAFQQZXDUAGAZzY2hlbWEVCAAVBCUCGAdzYWxl
X2lkABUMJQIYB2NvdW50cnklAEwcAAAAFQwlAhgFc3RhdGUlAEwcAAAAFQ4VChUCGAZhbW91bnQlChUEFRQsXBUEFRQAAAAW5AMZ
HBlMJgAcFQQZNQAGEBkYB3NhbGVfaWQVAhbkAxbIIxbgFCbYDyYIHBgIYhIBAAAAAAAYCHERAQAAAAAAFgAoCGISAQAAAAAAGAhx
EQEAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgDkAwAAACYAHBUMGTUABhAZGAdjb3VudHJ5FQIW5AMWgAMWhAMm1hUm6BQc
NgAoAlVTGAJBRRERABksFQQVABUCABUAFRAVAgA8FsgHGQYZJgDkAwAAACYAHBUMGTUABhAZGAVzdGF0ZRUCFuQDFpgDFpgDJuIY
JuwXHDaKASgCV0EYAkNBEREAGSwVBBUAFQIAFQAVEBUCADwWtAUZBhkmigHaAgAAACYAHBUOGTUABhAZGAZhbW91bnQVAhbkAxb6
FxaOGCaaLiaEGxwYBQAAWwpAGAUAAAevlBYAKAUAAFsKQBgFAAAHr5QREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA5AMAAAAW2kEW
5AMmCBaKMwAZHBgMQVJST1c6c2NoZW1hGIADLy8vLy94Z0JBQUFRQUFBQUFBQUtBQXdBQmdBRkFBZ0FDZ0FBQUFBQkJBQU1BQUFB
Q0FBSUFBQUFCQUFJQUFBQUJBQUFBQVFBQUFDc0FBQUFiQUFBQUVBQUFBQUVBQUFBZFAvLy93QUFBUWNRQUFBQUlBQUFBQVFBQUFB
QUFBQUFCZ0FBQUdGdGIzVnVkQUFBQ0FBTUFBUUFDQUFJQUFBQUNnQUFBQUlBQUFDcy8vLy9BQUFCQlJBQUFBQVlBQUFBQkFBQUFB
QUFBQUFGQUFBQWMzUmhkR1VBQUFEWS8vLy8xUC8vL3dBQUFRVVFBQUFBSEFBQUFBUUFBQUFBQUFBQUJ3QUFBR052ZFc1MGNua0FC
QUFFQUFRQUFBQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFJQUFBQUFRQUFBQUFBQUFBQndBQUFITmhiR1Zm
YVdRQUNBQU1BQWdBQndBSUFBQUFBQUFBQVVBQUFBQUFBQUFBABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZTBwA
ABwAABwAABwAAACqAwAAUEFSMQ==
""",
    "targets": """
UEFSMRUEFVQVUkwVDhUAEgAAKhQCAAAAVVMBBhxJTgIAAABTRwEGBEFFAQY0UUECAAAATEsCAAAATlAVABUcFSAsFRoVEBUGFQYc
NgAoAlVTGAJBRRERAAAADjQCAAAAGgEDBUi0gKkDABUEFWAVWEwVEBUAEgAAMBQCAAAAVFgFBgBOAQYETUgBBgRHSgEGBE5ZAQY0
S0ECAAAAREwCAAAAQ0EVABUYFRwsFRoVEBUGFQYcNgooAlRYGAJDQRERAAAADCwDAAAABWcZAwOIxvoVBBWQARWWAUwVGBUAEgAA
SPBHAAADk4cAAAAC+vCAAAAFXUqAAAAD39JAAAABycOAAAACrqVAAAAA5OHAAAAFqZXAAAACYloAAAABMS0AAAAELB2AAAAExLQA
FQAVIBUkLBUaFRAVBhUGHBgGAAAFqZXAGAYAAADk4cAWACgGAAAFqZXAGAYAAADk4cAREQAAABA8AgAAABoBBAUQEkNlh6kLABUE
GUw1ABgGc2NoZW1hFQYAFQwlAhgHY291bnRyeSUATBwAAAAVDCUCGAVzdGF0ZSUATBwAAAAVDhUMFQIYDXRhcmdldF9hbW91bnQl
ChUEFRgsXBUEFRgAAAAWGhkcGTwmABwVDBk1AAYQGRgHY291bnRyeRUCFhoWygEWzAEmdiYIHDYAKAJVUxgCQUUREQAZLBUEFQAV
AgAVABUQFQIAPBY0GQYZJgAaAAAAJgAcFQwZNQAGEBkYBXN0YXRlFQIWGhbSARbOASbIAibUARw2CigCVFgYAkNBEREAGSwVBBUA
FQIAFQAVEBUCADwWIBkGGSYKEAAAACYAHBUOGTUABhAZGA10YXJnZXRfYW1vdW50FQIWGha+AhbIAibYBCaiAxwYBgAABamVwBgG
AAAA5OHAFgAoBgAABamVwBgGAAAA5OHAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmABoAAAAW2gUWGiYIFuIFABkcGAxBUlJPVzpz
Y2hlbWEYuAIvLy8vLytBQUFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFBQUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFB
QUFBTUFBQUNFQUFBQVNBQUFBQVFBQUFDWS8vLy9BQUFCQnhBQUFBQW9BQUFBQkFBQUFBQUFBQUFOQUFBQWRHRnlaMlYwWDJGdGIz
VnVkQUFBQUFnQURBQUVBQWdBQ0FBQUFBd0FBQUFDQUFBQTJQLy8vd0FBQVFVUUFBQUFHQUFBQUFRQUFBQUFBQUFBQlFBQUFITjBZ
WFJsQUFBQXlQLy8veEFBRkFBSUFBWUFCd0FNQUFBQUVBQVFBQUFBQUFBQkJSQUFBQUFjQUFBQUJBQUFBQUFBQUFBSEFBQUFZMjkx
Ym5SeWVRQUVBQVFBQkFBQUFBPT0AGCBwYXJxdWV0LWNwcC1hcnJvdyB2ZXJzaW9uIDI1LjAuMRk8HAAAHAAAHAAAAOgCAABQQVIx
""",
}

_data_dir = pathlib.Path("data") / "018-join-on-nullable-key"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

sales = spark.read.parquet((_data_dir / "sales.parquet").resolve().as_posix())
sales.createOrReplaceTempView("sales")
targets = spark.read.parquet((_data_dir / "targets.parquet").resolve().as_posix())
targets.createOrReplaceTempView("targets")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'achieved': '7fb626ec327e89a7776c2d70ecc17a02e9db1b3270ab59f08c6c2728fcd79faf',
                         'actual_amount': 'c6ae15a9573de460108dd7ec077a692b4c2f00b152265ab71e092c03bc010d2a',
                         'country': '009609fc3ea4d7b92a43eedb3a319024aa6619d3b9193a4e110f50826c1c1879',
                         'state': 'ea656529bc1b53fa93916c8fddb0907a0378ed09374c51da6e49edc8c375f599',
                         'target_amount': 'e87253ba036dbae97d310619f7b29637ac60c1f973a3abd7d5a575fe78f04ff9'},
 'columns': [['achieved', 'boolean'],
             ['actual_amount', 'decimal(14,2)'],
             ['country', 'string'],
             ['state', 'string'],
             ['target_amount', 'decimal(12,2)']],
 'fingerprint': 'f18c3843ce4f6c06385f37991b72a84f30bad8c245b11787343db75df1a56260',
 'order_matters': False,
 'row_count': 13}

check = make_check(EXPECTED)

## The situation

Sales targets are set per region, and a region is a country plus a state. Small
countries are managed as one region and have no state.

An AI assistant wrote the target report in the answer cell. It runs, returns one row
per target, and the big regions look right. But the managers for Singapore and the
UAE are being told they sold nothing this quarter, and they have the invoices to
prove otherwise.

Find the bug and fix it.

## Input tables

### `sales`

One row per sale.

| Column | Type | Description |
|---|---|---|
| `sale_id` | BIGINT | Unique per sale. |
| `country` | STRING | Country code. Never NULL. |
| `state` | STRING | State code. NULL for countries that are managed as a single region. |
| `amount` | DECIMAL(10,2) | Sale value in rupees. |

### `targets`

One row per region that has a target.

| Column | Type | Description |
|---|---|---|
| `country` | STRING | Country code. Never NULL. |
| `state` | STRING | State code. NULL for countries that are managed as a single region. |
| `target_amount` | DECIMAL(12,2) | The region's sales target for the quarter. |

## Requirement

Return every region in `targets` with its actual sales.

- A sale belongs to a region if the country is the same and the state is the same.
  For a single-region country, the state is NULL on both sides and that counts as
  the same.
- `actual_amount` is the sum of the region's sales, and 0.00 if it has none.
- `achieved` is true if `actual_amount` is at least `target_amount`.
- Sales in regions without a target are ignored.

## Expected output

One row per region in `targets`.

| Column | Type | Description |
|---|---|---|
| `country` | STRING | |
| `state` | STRING | NULL for single-region countries. |
| `target_amount` | DECIMAL(12,2) | |
| `actual_amount` | DECIMAL(14,2) | 0.00 if the region has no sales. |
| `achieved` | BOOLEAN | |

In [ ]:
sales.show(5)
targets.show(5)

## Your answer

The cell below is the AI-written code. It runs, but its output is wrong. Find the bug and fix it. The tables are available as DataFrames and as views: `sales`, `targets`.

In [ ]:
from pyspark.sql import functions as F


def solve(spark):
    """Actual sales against target for every region."""
    sales = spark.table("sales")
    targets = spark.table("targets")

    # Total sales per region
    actuals = sales.groupBy("country", "state").agg(
        F.sum("amount").alias("actual_amount")
    )

    # Left join so that regions without any sales are kept
    report = targets.join(actuals, on=["country", "state"], how="left")

    # Regions without sales have an actual of zero
    report = report.withColumn(
        "actual_amount",
        F.coalesce(F.col("actual_amount"), F.lit(0)).cast("decimal(14,2)"),
    )

    # Flag the regions that reached their target
    result = report.select(
        "country",
        "state",
        "target_amount",
        "actual_amount",
        (F.col("actual_amount") >= F.col("target_amount")).alias("achieved"),
    )

    return result


result = solve(spark)
result.show()

In [ ]:
check(result)